# File Operations and Change Checks

CSC-239 · Module 8 · Lesson 3 of 3

You can now create text fixtures and query their contents. This lesson manages those files and compares successive reads so a program can notice an observed state change.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Copy, move, inspect and delete only files created by the exercise.
- Compare successive text snapshots and explain what a polling check can miss.


## Why This Matters

A game may exchange state through a shared file. A player needs to read changes accurately, while development tools may need backups, renamed files, and deliberate cleanup.


## Check Your Starting Point

Explain a Path, UTF-8 whole-file reading, String.equals, a checked IOException contract, and a class implementing an interface. Recall that a successful read gives you a String value at that moment.

**My explanation:**


## Concept

### Observe existence and type

A **file existence and type check** asks whether a path currently exists or currently names a regular file. Files.exists and Files.isRegularFile return boolean observations. A regular file holds ordinary file data, unlike a directory that groups entries.

These checks do not promise that a later operation will succeed. Another program can change the path after the check, and false can also mean the operation could not determine the answer. Handle IOException from the actual read or write instead of treating an earlier true result as a guarantee.

### Copy or move a private fixture

A **file copy** creates another file with the source data while retaining the original. A **file move** relocates or renames a file so its old path no longer names it after success.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-copy-");
    Path source = directory.resolve("source.txt");
    Path copy = directory.resolve("copy.txt");
    Path moved = directory.resolve("moved.txt");
    Files.writeString(source, "map", StandardCharsets.UTF_8);
    Files.copy(source, copy);
    Files.move(copy, moved);
    System.out.println("Source: " + Files.exists(source));
    System.out.println("Old copy: " + Files.exists(copy));
    System.out.println("Moved file: " + Files.isRegularFile(moved));
    System.out.println(Files.readString(moved, StandardCharsets.UTF_8));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

This prints Source: true, Old copy: false, Moved file: true, and map. Copy keeps source. Move changes the path of the copied file.

The target paths start unused because this example created a new directory. Default copy or move can fail when the target already exists; do not assume it silently replaces an existing file. Keep these operations inside your own fixture directory and choose replacement behavior explicitly when an application actually requires it.

### Delete what the exercise created

**File deletion** removes a file or an empty directory entry. Files.deleteIfExists returns true when it removed the entry and false when the entry was already absent. Other failures can still raise IOException.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-clean-");
    Path file = directory.resolve("note.txt");
    Files.writeString(file, "practice", StandardCharsets.UTF_8);
    System.out.println("Removed file: " + Files.deleteIfExists(file));
    System.out.println("Removed again: " + Files.deleteIfExists(file));
    System.out.println("Removed directory: " + Files.deleteIfExists(directory));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

This prints Removed file: true, Removed again: false, and Removed directory: true. Delete the known file before deleting its directory. Removing a nonempty directory this way fails.

Earlier lesson fixtures remain in temporary storage until cleanup. These examples remove only the exact files and directory they created. If an earlier operation fails, normal cleanup statements after it can be skipped. For long-lived applications, design cleanup around the failure paths you learned in Module 7 and preserve any primary exception.

### Inspect metadata without confusing it with content

**File metadata** is information about a file apart from the text it contains. Files.size returns a byte count with Java's **long** type. Like int, long is a primitive type for whole numbers, but it has a larger range. We store the result in a long variable so it keeps the type returned by Files.size. Files.getLastModifiedTime returns a FileTime object representing the recorded last-modified time. You can display that time for inspection or compare it with another observed FileTime using equals.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-meta-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "open", StandardCharsets.UTF_8);
    long beforeSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "busy", StandardCharsets.UTF_8);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Same byte size: " + (beforeSize == Files.size(file)));
    System.out.println("Timestamp observed: " + (beforeTime != null && afterTime != null));
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

This prints Same byte size: true and Timestamp observed: true. Both values use four plain English letters and occupy four UTF-8 bytes, yet their contents differ.

The example confirms that we obtained timestamps; it does not claim they differ. File systems differ in how small a time interval they can record, so rapid writes may share the same recorded time. A timestamp comparison is an observation to inspect, not a guaranteed detector of every rewrite. Size and last-modified time are useful clues, but neither alone proves the file's text stayed unchanged.

### Compare content snapshots

A **content snapshot** is a retained read of the file used as a comparison baseline. Read the current text and use equals to compare it with the previous String. This detects differing successful reads even when the file has the same byte size.

**Polling** means checking state at separate moments to notice changes between observations. Our independent tutorial helper performs one check each time the caller invokes check. It stores the current read as the baseline for the next successful check.

Use this order:

1. Read current text. If reading fails, report IOException.
2. Compare current text with the previous snapshot.
3. Store current text as the new snapshot.
4. Return whether the values differed.

Updating the baseline after a failed read would lose useful state. Returning false after catching and ignoring IOException would falsely claim no change. Let the caller distinguish a failed observation from a successful unchanged observation.

### State what polling can miss

Suppose your earlier snapshot is A. Another program writes B and then A before the next check. The next successful read equals the previous snapshot, so this checker reports false. It cannot know about the unseen intermediate B.

Reading while another program writes does not ensure that the writer has finished all its intended changes. Comparing snapshots does not prevent that other program from changing the file during your read. Follow the game's supplied communication rules for real shared files.

We call the checker a few times explicitly. Each call observes the file as it is available at that moment.

The tutorial ChangeProbe and TextChangeProbe define a small practice contract. Your assignment's FileManager must implement its supplied FileTextReader and FileTextWriter interfaces and extend its supplied AbstractFileMonitor. Use those actual method declarations and rules; do not guess them by renaming tutorial methods.


## Video Demonstration

Compare file paths after copy and move, then compare both size and text after a rewrite. Predict the boolean observations and the two deletion results before execution.

<video controls preload="metadata" width="960">
  <source src="media/03_file_operations_and_change_checks/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_file_operations_and_change_checks/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the file operations and change checks demonstration transcript](media/03_file_operations_and_change_checks/transcript.md).


## Worked Example

**Subgoal 1: make a backup and rename it.** Keep the source, copy it, and move the copied entry to archive.txt.

**Subgoal 2: compare observations.** Read a baseline, replace ready with a different text of the same byte length, and compare size and contents separately.

**Subgoal 3: clean the fixture.** Remove the archive, observe that a second removal has no entry to remove, then delete the remaining source and empty directory.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "ready\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "busy!\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: true
Deleted archive: true
Deleted again: false
```

The source remains after the copy. The backup path disappears after the move, while the archive names a regular file. The two text versions have equal byte size but different content. The first archive removal returns true and the second returns false. All affected paths belong to the newly created practice directory.


## Predict, Run, Trace, and Explain

### Predict paths, size and content

Before running, predict all seven printed lines. Track which of `state.txt`, `backup.txt` and `archive.txt` exist after copy and after move. The source initially holds `idle\n` and later holds `playing\n`. Count their stored bytes for these letters and newline, then predict the size and content comparisons. Explain why deleting the archive twice can give different results and why the source file must be removed before its directory.

My predicted seven lines:

Paths after copy and after move:

Byte counts and comparison reasoning:

Why the deletion results differ:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Run the complete program in the Java kernel of your Workspace. Compare all seven lines with your prediction and explain any revision without erasing your original answer. Account for each text’s byte count and both deletion results. Run the whole cell again and explain why its new directory makes the paths available for a fresh copy and move. Describe which files and directory the successful run removes.

My original prediction:

Actual output:

What matched or changed, and why:

Where the byte counts come from:

Replay result and the entries this run removes:

### Trace file locations and successful reads

Trace the file entries after the initial write, copy, move, rewrite and cleanup. Distinguish the source entry from the copied entry. Explain why rewriting `state.txt` does not itself rewrite the archived copy. Mark where `previous` and `current` are read and which stored String each comparison uses. Explain why a successful `Files.exists` observation does not guarantee that a later read succeeds, and why the actual operations still need the `IOException` handler.

Operation | Existing entries | Text associated with each entry

My file-state trace:

When previous and current are captured:

Why a path check cannot guarantee the next operation:

What happens if an operation reaches the handler:

<details>
<summary>Show answer</summary>

The first three reports are true, false and true: the source remains, the backup path no longer exists after the move, and the archive is a regular file. The supplied text changes from 5 to 8 UTF-8 bytes, producing Same size false and Content changed true. Deleting the archive reports true and then false. The remaining source is removed before the empty directory. A complete successful replay creates a new directory, so it does not reuse earlier copy/move targets. Its same supplied operations produce the same seven reports. After the first write only the source exists. Copy adds a second file with the initial text; move changes that copied file’s location from backup to archive. Rewriting the source changes its text to `playing\n`; the archive still contains the earlier copied text. The previous String was read before the rewrite and current afterward. The program compares their contents and observes the current byte size separately. Cleanup removes the archive, then the source, then the empty directory. A path observation describes that moment. Another program could change the path before a later operation, and a false existence/type result can also mean the answer could not be determined. The actual read, copy, move or delete can still raise `IOException`. On failure, later statements in that `try`, including normal cleanup, may be skipped.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: false
Content changed: true
Deleted archive: true
Deleted again: false
```

Common error: Treating copy as removal of the source. Expecting the old backup path to keep naming the moved file. Ignoring the newline in these byte counts. Expecting both deletion calls to remove the same entry.

</details>


### Inspect byte counts and recorded times

Predict all six printed lines, then run this complete program. It rewrites `café` as `tea` and records a `FileTime` before and after the rewrite. A `FileTime` represents the file’s recorded last-modified time. Compare the recovered String length with the stored byte count. Explain what `Times obtained:` proves and what it does not prove about the two times. Could two rapid writes have the same recorded time? Explain why content comparison supplies different evidence from metadata. Keep every operation inside the new directory.

My predicted six lines:

Actual output:

Why café has different String-length and byte-count results:

What the two FileTime values represent:

What Times obtained establishes and leaves unknown:

Why content comparison adds evidence:


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-metadata-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "café", StandardCharsets.UTF_8);
    String previous = Files.readString(file, StandardCharsets.UTF_8);
    long previousSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "tea", StandardCharsets.UTF_8);
    String current = Files.readString(file, StandardCharsets.UTF_8);
    long currentSize = Files.size(file);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Before characters: " + previous.length());
    System.out.println("Before bytes: " + previousSize);
    System.out.println("After bytes: " + currentSize);
    System.out.println("Times obtained: " + (beforeTime != null && afterTime != null));
    System.out.println("Same byte size: " + (previousSize == currentSize));
    System.out.println("Content changed: " + !current.equals(previous));
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The recovered `café` String has length 4 but occupies 5 UTF-8 bytes; `tea` occupies 3 bytes. The six reports are 4, 5, 3, true, false and true. `Files.getLastModifiedTime` returns the recorded time as a `FileTime`; the program obtains one before and one after writing. The non-null check confirms both values were obtained. It does not compare the times or promise they differ. A file system may record two rapid writes with the same time. Size and recorded time are metadata, separate from the text. The String comparison establishes that these two successful reads differ. The program removes its known file before removing its empty directory.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.attribute.FileTime;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-metadata-");
    Path file = directory.resolve("state.txt");
    Files.writeString(file, "café", StandardCharsets.UTF_8);
    String previous = Files.readString(file, StandardCharsets.UTF_8);
    long previousSize = Files.size(file);
    FileTime beforeTime = Files.getLastModifiedTime(file);
    Files.writeString(file, "tea", StandardCharsets.UTF_8);
    String current = Files.readString(file, StandardCharsets.UTF_8);
    long currentSize = Files.size(file);
    FileTime afterTime = Files.getLastModifiedTime(file);
    System.out.println("Before characters: " + previous.length());
    System.out.println("Before bytes: " + previousSize);
    System.out.println("After bytes: " + currentSize);
    System.out.println("Times obtained: " + (beforeTime != null && afterTime != null));
    System.out.println("Same byte size: " + (previousSize == currentSize));
    System.out.println("Content changed: " + !current.equals(previous));
    Files.deleteIfExists(file);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Before characters: 4
Before bytes: 5
After bytes: 3
Times obtained: true
Same byte size: false
Content changed: true
```

Common error: Using String length as the stored byte size. Reading Times obtained as proof that the two times differ. Assuming a recorded time can prove that every rewrite was observed.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete copy, move, inspection and deletion

The draft is incomplete and shown for editing. Replace `COPY_OPERATION`, `MOVE_OPERATION`, `TYPE_OPERATION` and `DELETE_OPERATION` with `copy`, `move`, `isRegularFile` and `deleteIfExists`, once each. Preserve the new directory, all fixed file names, the operation order and the later cleanup. Copy the completed program into the empty work cell. Predict the seven reports, then run it. Explain which operation retains its source, which changes the copied file’s path, and why the directory deletion belongs last.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.COPY_OPERATION(state, backup);
    Files.MOVE_OPERATION(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.TYPE_OPERATION(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.DELETE_OPERATION(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


My four replacements and purposes:

Predicted output:

Actual output:

How copy and move differ:

Why fresh targets and cleanup order matter:

<details>
<summary>Show answer</summary>

Use copy, move, isRegularFile and deleteIfExists in that order. Copy retains the source; move relocates the copied entry from backup to archive. The type check asks whether archive names a regular file at that moment. The first deletion removes archive; the supplied second deletion reports its absence. The source must then be removed before the directory is empty. The targets begin unused inside a new directory. Default copy/move can fail when a target already exists, so rerun the complete setup rather than assuming an old target will be replaced.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: false
Content changed: true
Deleted archive: true
Deleted again: false
```

Common error: Swapping source and destination arguments while filling a method name. Replacing a type observation with a file-changing operation. Deleting the directory before its known files. Assuming default copy or move silently replaces an existing target.

</details>


### Compare rewriting the same text with changing it

Change only the later write in the working starter so it writes `idle\n` again. Keep the initial text and all file operations unchanged. Predict the size and content reports, then run the complete modified cell and explain them. Next test a complete version that rewrites the source as `busy\n`. Compare its two reports with the same-content version. Explain why a write operation and a difference between observed Strings are not the same event.


In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "playing\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}


Same-content prediction and actual reports:

Same-size-different-content prediction and actual reports:

Why a write can occur without an observed content change:

Why equal byte size is insufficient:

<details>
<summary>Show answer</summary>

Rewriting `idle\n` leaves the recovered content and its 5-byte size unchanged, so the reports are Same size true and Content changed false. A write occurred, but the two snapshots are equal. Rewriting as `busy\n` also uses 5 bytes, yet the text differs from `idle\n`; that case reports true for both Same size and Content changed. Equal byte size alone cannot establish equal text. Copy, move and cleanup still operate on only this run’s new files.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: false
Deleted archive: true
Deleted again: false
```

Common error: Returning changed merely because a write statement ran. Treating equal byte sizes as equal content. Changing the initial snapshot as well as the later write, making a different test.

**Additional test: `Rewrite idle and a newline as busy and a newline`.** Both Strings occupy 5 UTF-8 bytes, but they differ. This complete program demonstrates true for Same size and true for Content changed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
try {
    Path directory = Files.createTempDirectory("csc239-state-");
    Path state = directory.resolve("state.txt");
    Path backup = directory.resolve("backup.txt");
    Path archive = directory.resolve("archive.txt");
    Files.writeString(state, "idle\n", StandardCharsets.UTF_8);
    Files.copy(state, backup);
    Files.move(backup, archive);
    System.out.println("Source exists: " + Files.exists(state));
    System.out.println("Backup exists: " + Files.exists(backup));
    System.out.println("Archive is file: " + Files.isRegularFile(archive));
    String previous = Files.readString(state, StandardCharsets.UTF_8);
    long previousSize = Files.size(state);
    Files.writeString(state, "busy\n", StandardCharsets.UTF_8);
    String current = Files.readString(state, StandardCharsets.UTF_8);
    System.out.println("Same size: " + (previousSize == Files.size(state)));
    System.out.println("Content changed: " + !current.equals(previous));
    System.out.println("Deleted archive: " + Files.deleteIfExists(archive));
    System.out.println("Deleted again: " + Files.deleteIfExists(archive));
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Source exists: true
Backup exists: false
Archive is file: true
Same size: true
Content changed: true
Deleted archive: true
Deleted again: false
```

</details>


### Repair snapshot update order

The faulty `SnapshotCheck` should report whether the current read differs from the previous successful read. Predict its three reports when the file changes from `cold` to `warm`. Trace `previous`, `current` and `changed` inside the faulty method. Reorder only the snapshot assignment and comparison so the comparison uses the earlier snapshot. Put the complete repaired program in the empty work cell and run it. Explain why the changed text is reported once and the repeated check reports no change. Test a complete version that rewrites `cold` as `cold` instead.

This sample is for repair:

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        previous = current;
        boolean changed = !current.equals(previous);
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "warm", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```


My predicted faulty output:

Faulty previous/current/changed trace:

My repaired trace and actual output:

Why the repeat check is unchanged:

Same-content prediction and actual output:

<details>
<summary>Show answer</summary>

The faulty method sets previous to current before comparing them, so both names describe equal text and every result is false. Compute changed first, then store current as the snapshot for the next successful check. The initial cold read is unchanged, warm differs from cold, and the next warm read matches the updated snapshot: false, true, false. The same-content test reports false three times. Updating the snapshot after the comparison is still necessary; otherwise repeated checks would keep comparing against the constructor’s old text.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "warm", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial: false
After rewrite: true
Repeat: false
```

Common error: Always returning true after a read instead of comparing text. Removing the snapshot update entirely. Keeping the assignment before the comparison. Expecting another change on a repeat read with no rewrite.

**Additional test: `Repaired checker with cold rewritten as cold`.** All three observations are unchanged. This checks that the repair detects text differences, not the fact that a write statement was executed.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
class SnapshotCheck {
    private Path file;
    private String previous;
    public SnapshotCheck(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-snapshot-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    SnapshotCheck checker = new SnapshotCheck(state);
    System.out.println("Initial: " + checker.check());
    Files.writeString(state, "cold", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + checker.check());
    System.out.println("Repeat: " + checker.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial: false
After rewrite: false
Repeat: false
```

</details>


## Independent Practice

### Construct the tutorial text change checker

Define interface `ChangeProbe` with `boolean check() throws IOException`. Implement `TextChangeProbe` with a constructor that receives a `Path`, stores it, and reads the initial UTF-8 text as its snapshot. Each successful check must read current text, compare it with the previous snapshot, store the current snapshot, and return whether the two reads differ. Declare `IOException` so the caller can distinguish a failed read from an unchanged observation.

Create `state.txt` with text `open` in a new temporary directory using prefix `csc239-probe-`. Construct a `TextChangeProbe` and use it through a `ChangeProbe` variable. Check once, rewrite `busy` and check, check again without rewriting, then rewrite `""` and check. Print the results with `Initial change: `, `After rewrite: `, `Without rewrite: ` and `After empty: ` in that order. Include all imports and an outer `IOException` handler that prints `File problem: ` plus the message. Delete only this example’s state file and then its empty directory. Predict the four results before running and explain the snapshot used by each check afterward.

These tutorial types do not replace the instructor’s `AbstractFileMonitor` or `FileManager` declarations.

My predicted four reports:

Actual reports:

Snapshot before and after each successful check:

Why empty text is a successful read:

How a failed read differs from a false result:


### Test unchanged writes, failed reads and missed changes

Keep your `ChangeProbe` and `TextChangeProbe` implementations unchanged. Run each case as a complete program with a new directory. Predict the labeled results first, record the actual results, and explain each comparison.

1. In the original sequence, replace only the `busy` write with another `open` write. Keep the later empty write.
2. Create `open`, construct the probe, then remove the state file before calling `check`. Put that call in a nested `try` that would print `Unexpected result: ` plus the result if it returned. Catch `IOException` there and print `Missing file reported.`. Recreate the same state path with its original text `open`, then print `After recovery: ` with another check. Next write `busy` and print `After rewrite: ` with a further check. Clean up the recreated file and its directory. Explain how restoring the original text tests whether the earlier successful snapshot survived the failed read.
3. Create text `A` and construct a new probe. Write `B` and then `A` without a check between them. Print `After A-B-A: ` with one check, then clean up.
4. Repeat the A/B/A setup, but check after writing B and again after writing A. Print `After B: ` and `After A: `, then clean up.

Explain why the last two cases differ. State what a successful false result tells you and why it cannot establish that no write occurred between checks.

Case | Predicted results | Actual results | Snapshot explanation

Same-content rewrite and later empty text:

Removed file, original-content recovery and later different text:

A/B/A with one later check:

Checks after both B and A:

What false establishes and what it cannot establish:


<details>
<summary>Show answer</summary>

The constructor captures open. The first check reads open and returns false. The busy read differs, so it returns true and stores busy. The next busy read matches and returns false. Reading empty text differs from busy, so the final result is true and the snapshot becomes empty. The method stores a new snapshot only after a successful read and comparison. The interface and implementation declare IOException, leaving failure handling with the caller. Successful cleanup removes the known file before its empty directory. The original four reports provide the baseline below. Rewriting open as open reports no change, while the later empty text still differs. Removing the file makes readString throw IOException, so the missing-file test reports a failure instead of a boolean result. The previous successful snapshot should remain open. Restoring open must therefore report false; this checks preservation directly. Writing busy afterward must report true, confirming that a later different read is still detected. In the unobserved A/B/A case, the next read is A, equal to the saved A, so the result is false even though B was written. When checks occur after both writes, B differs from A and then A differs from B, so both results are true. Polling compares separate successful reads; it cannot recover an intermediate value that was never read.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    System.out.println("Initial change: " + probe.check());
    Files.writeString(state, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    System.out.println("Without rewrite: " + probe.check());
    Files.writeString(state, "", StandardCharsets.UTF_8);
    System.out.println("After empty: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial change: false
After rewrite: true
Without rewrite: false
After empty: true
```

Common error: Comparing every check only with the constructor’s original text. Updating the snapshot before computing the result. Catching a failed read and returning false as though it had succeeded. Removing a directory before its file.

**Additional test: Rewrite open as open before the empty-text step.** The same-content rewrite is unchanged. The repeat also stays unchanged; empty text later differs from open.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    System.out.println("Initial change: " + probe.check());
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    System.out.println("Without rewrite: " + probe.check());
    Files.writeString(state, "", StandardCharsets.UTF_8);
    System.out.println("After empty: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Initial change: false
After rewrite: false
Without rewrite: false
After empty: true
```

**Additional test: Remove the file, restore open, then change it to busy.** The failed read returns no boolean. Restoring the original open text must report unchanged, directly checking that the last successful snapshot survived. The following busy rewrite must report a change. The unexpected-result line must not appear.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.deleteIfExists(state);
    try {
        System.out.println("Unexpected result: " + probe.check());
    } catch (IOException problem) {
        System.out.println("Missing file reported.");
    }
    Files.writeString(state, "open", StandardCharsets.UTF_8);
    System.out.println("After recovery: " + probe.check());
    Files.writeString(state, "busy", StandardCharsets.UTF_8);
    System.out.println("After rewrite: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
Missing file reported.
After recovery: false
After rewrite: true
```

**Additional test: Write B and return to A before checking.** The only new observation is A, equal to the saved A. The program really performs both writes, but this checker never reads B.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.writeString(state, "B", StandardCharsets.UTF_8);
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    System.out.println("After A-B-A: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
After A-B-A: false
```

**Additional test: Check after B and after the return to A.** The first successful check stores B after finding a difference from A. The next check compares A with B, so both observed changes are reported.

```java
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.nio.charset.StandardCharsets;
import java.io.IOException;
interface ChangeProbe {
    boolean check() throws IOException;
}
class TextChangeProbe implements ChangeProbe {
    private Path file;
    private String previous;
    public TextChangeProbe(Path file) throws IOException {
        this.file = file;
        previous = Files.readString(file, StandardCharsets.UTF_8);
    }
    @Override
    public boolean check() throws IOException {
        String current = Files.readString(file, StandardCharsets.UTF_8);
        boolean changed = !current.equals(previous);
        previous = current;
        return changed;
    }
}
try {
    Path directory = Files.createTempDirectory("csc239-probe-");
    Path state = directory.resolve("state.txt");
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    ChangeProbe probe = new TextChangeProbe(state);
    Files.writeString(state, "B", StandardCharsets.UTF_8);
    System.out.println("After B: " + probe.check());
    Files.writeString(state, "A", StandardCharsets.UTF_8);
    System.out.println("After A: " + probe.check());
    Files.deleteIfExists(state);
    Files.deleteIfExists(directory);
} catch (IOException problem) {
    System.out.println("File problem: " + problem.getMessage());
}
```

Expected output:

```text
After B: true
After A: true
```

</details>


## Summary

File checks describe current observations. Copy retains its source, move changes a path, and deletion removes an entry. Metadata describes properties such as size and time; it is not the content itself. A snapshot checker compares successive successful reads and updates its baseline. Polling can miss intermediate changes and does not prevent another program from changing the file during a read.

Close the answers. Explain a same-size content change and an A-to-B-to-A sequence that polling misses.


## Reflection

A game-state file may change between two reads. Describe the evidence your player has after a successful check, what remains unknown, and how it should distinguish an I/O failure from no observed change.

**My design and explanation:**

Module 9 uses the file and dictionary mechanisms in the Ghost project. Its project guide and Canvas pages provide milestones and discussion; no notebook is required.


## Supplemental Reading

- [Java 21 Files API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) documents copy, move, delete, checks, metadata, and text reads.
- [Java 21 FileTime API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/attribute/FileTime.html) describes file timestamps and comparisons.
- [Java 21 Path API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Path.html) explains the locations used by file operations.
